In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

cleaned_path = OUT_DIR / "cleaned_original.csv"

# Let pandas detect the delimiter
data = pd.read_csv(cleaned_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data


In [ ]:
# Expand truncated output to print all the value counts
s = data['sdo2_orientation_Event_Types_Attended'].value_counts(dropna=False)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(s)          # or: display(s) in Jupyter


In [ ]:
data.columns

# Handle low frequency categories in columns

sdo5_degree_degree

sdo1_previous_Previous_Education_Type

sdo2_skc_ADVIES_DEF

sdo2_orientation_Event_Types_Attended

  (see cleaning data script for detalis)

In [ ]:
# -----------------------------------------------------------------------------
# Handling Categorical Variables for Mixed Model Families
#
# Purpose
#   Merge low-frequency categories into "Others" for selected categorical columns
#   to reduce noise and overfitting, while preserving interpretability.
#
# Background and Rationale
#   • Original statistical cutoff: 0.5% of dataset
#         N = 47,946 rows
#         p = 0.005  →  minimum_count = N × p = 47,946 × 0.005 ≈ 240
#     This threshold ensures each category appears ~48 times per fold in 5-fold
#     cross-validation, promoting model stability and avoiding tiny groups.
#
#   • Business adjustment:
#     Some rare categories (e.g., specific degree programs) have strong
#     operational or policy relevance. To retain these, the minimum category
#     size is relaxed to 100 rows.
#
#   • New threshold:
#         minimum_count = 100  →  p = 100 / 47,946 ≈ 0.0021 (≈0.21%)
#
# Decision Summary
#   Statistically recommended: 240 (≈0.5%) — optimal for stability and CV balance.
#   Business decision: 100 (≈0.2%) — retains meaningful but smaller categories.
#
# Notes
#   • NaN values are preserved (not merged into "Others").
#   • Only selected categorical columns are affected.
# -----------------------------------------------------------------------------

# Columns to apply the cutoff to
cat_cols = [
    "sdo5_degree_degree",
    "sdo1_previous_Previous_Education_Type",
    "sdo2_skc_ADVIES_DEF",
    "sdo2_orientation_Event_Types_Attended"
]

N = 47946

# --- Original statistical cutoff (for reference)
p_statistical = 0.005
min_count_statistical = ceil(N * p_statistical)  # ≈ 240

# --- Relaxed business cutoff
min_count = 100

print(f"Statistical cutoff (0.5%): {min_count_statistical}")
print(f"Business cutoff (~0.2%): {min_count}")

# Apply cutoff using business threshold
for col in cat_cols:
    vc = data[col].value_counts(dropna=False)
    rare_levels = [lvl for lvl in vc.index if (not pd.isna(lvl)) and (vc[lvl] < min_count)]
    if rare_levels:
        data[col] = data[col].where(~data[col].isin(rare_levels), "Others")


In [ ]:
print(data['sdo5_degree_degree'].value_counts())

In [ ]:
# -----------------------------------------------------------------------------
# Result:
#   Only the following categories from sdo5_degree_degree were affected by the 
#   frequency cutoff (each had fewer than 100 occurrences and was merged into "Others"):
#
#   Drama_Therapy......................... 89
#   Teacher............................... 80
#   Teacher_Education_German.............. 80
#   Teacher_Education_Physics............. 77
#   Teacher_Education_Chemistry........... 73
#   Teacher_Education_French.............. 54
#   Teacher_Education_Technology.......... 24
# -----------------------------------------------------------------------------


In [ ]:
print(data['sdo1_previous_Previous_Education_Type'].value_counts())

In [ ]:
print(data['sdo2_skc_ADVIES_DEF'].value_counts())

In [ ]:
print(data['sdo2_orientation_Event_Types_Attended'].value_counts())

In [ ]:
data.columns

In [ ]:
data.to_csv(f"{OUT_DIR}/handled_low_frequency_categories.csv", index=False)
print(f"✅ Saved to: {OUT_DIR}/handled_low_frequency_categories.csv")
